# TrumorGPT with Qwen2.5 on Google Colab

**IEEE 2025 Paper Replication**: "TrumorGPT: Topic-Enhanced GraphRAG for Health Misinformation Detection"

This notebook runs the complete TrumorGPT pipeline using **Qwen2.5-7B-Instruct (4-bit quantized)** instead of Gemini API.

## Key Features
- ✅ Local LLM inference (no API quota limits)
- ✅ 4-bit quantization for Qwen2.5-7B-Instruct (fits in Colab GPU)
- ✅ Falls back to Qwen2.5-3B-Instruct if VRAM insufficient
- ✅ Complete GraphRAG pipeline (retrieval → verification → explanation)
- ✅ Crash-safe checkpointing to Google Drive
- ✅ Resume after Colab disconnections
- ✅ Separate experiment directories from Gemini results

## Prerequisites
1. **Google Colab Pro/Pro+** recommended (for A100/T4 with 16GB+ VRAM)
2. **Google Drive** mounted for persistent checkpoints
3. **GitHub repository** with TrumorGPT code

## Quick Start
1. Open this notebook in Google Colab
2. Run **Cell 1** (Setup & GPU Check)
3. Run **Cell 2** (Mount Google Drive)
4. Run **Cell 3** (Clone Repository & Install Dependencies)
5. Run **Cell 4** (Download & Load Qwen2.5 Model)
6. Run **Cell 5** (Initialize Pipeline & KB)
7. Run **Cell 6** (Test on 3 Diagnostic Claims)
8. Run **Cell 7** (Evaluate on 10 Dev Claims)
9. Review results → **Approve to proceed**
10. Run **Cell 8** (Evaluate on 100 Dev Claims)
11. Review results → **Approve to proceed**
12. Run **Cell 9** (Evaluate on 400 Test Claims - held-out)

In [ ]:
# =============================================================================
# CELL 1: SETUP & GPU CHECK
# =============================================================================
# Run this first to verify GPU availability and VRAM

import torch
import os
import sys
import subprocess

print("=" * 70)
print("GPU AVAILABILITY CHECK")
print("=" * 70)

# Check CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"  GPU {i}: {props.name} - {vram_gb:.1f} GB VRAM")
else:
    print("\u26a0\ufe0f  No GPU detected! Switch to GPU runtime: Runtime -> Change runtime type -> GPU")

# Check transformers & bitsandbytes
try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
except ImportError:
    print("Transformers not installed - will install in Cell 3")

try:
    import bitsandbytes
    print(f"BitsAndBytes version: {bitsandbytes.__version__}")
except ImportError:
    print("BitsAndBytes not installed - will install in Cell 3")

# Determine model size based on VRAM
if torch.cuda.is_available():
    max_vram = max(torch.cuda.get_device_properties(i).total_memory / (1024**3) 
                   for i in range(torch.cuda.device_count()))
    if max_vram >= 14:
        RECOMMENDED_MODEL = "Qwen/Qwen2.5-7B-Instruct"
        QUANTIZATION = "4bit"
    elif max_vram >= 8:
        RECOMMENDED_MODEL = "Qwen/Qwen2.5-7B-Instruct"
        QUANTIZATION = "4bit"
        print("\n\u26a0\ufe0f  VRAM may be tight for 7B 4-bit. Consider 3B if OOM.")
    else:
        RECOMMENDED_MODEL = "Qwen/Qwen2.5-3B-Instruct"
        QUANTIZATION = "4bit"
    print(f"\n\u2705 Recommended model: {RECOMMENDED_MODEL} ({QUANTIZATION})")
else:
    RECOMMENDED_MODEL = "Qwen/Qwen2.5-3B-Instruct"
    QUANTIZATION = "4bit"
    print(f"\n\u26a0\ufe0f  No GPU - using CPU fallback: {RECOMMENDED_MODEL}")

print("=" * 70)

In [ ]:
# =============================================================================
# CELL 2: MOUNT GOOGLE DRIVE
# =============================================================================
# Required for persistent checkpoints across Colab disconnections

from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define paths
DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_DIR = '/content/trumourgpt'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/trumourgpt_checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/trumourgpt_results'
CACHE_DIR = f'{DRIVE_ROOT}/trumourgpt_cache'

# Create directories
for d in [CHECKPOINT_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"Created: {d}")

print("\u2705 Google Drive mounted and directories created")

In [ ]:
# =============================================================================
# CELL 3: CLONE REPOSITORY & INSTALL DEPENDENCIES
# =============================================================================
# Replace with your GitHub repository URL

REPO_URL = "https://github.com/YOUR_USERNAME/trumourgpt.git"  # ← UPDATE THIS

# Clone repository
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"Repository already exists at {PROJECT_DIR}")
    # Pull latest changes
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# Install dependencies
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Install additional Colab requirements
colab_requirements = [
    "bitsandbytes>=0.43.0",
    "accelerate>=0.30.0",
    "transformers>=4.40.0",
    "torch>=2.3.0",
]
for pkg in colab_requirements:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# Install Ollama for embeddings (if not present)
try:
    import ollama
    print("Ollama Python client available")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ollama"], check=True)

print("\u2705 Dependencies installed")

In [ ]:
# =============================================================================
# CELL 4: DOWNLOAD & LOAD QWEN2.5 MODEL
# =============================================================================
# This loads the model with 4-bit quantization

from trumourgpt.llm.qwen_client import QwenClient, QwenClientConfig

# Configure Qwen client
qwen_config = QwenClientConfig(
    model_name=RECOMMENDED_MODEL,
    fallback_model="Qwen/Qwen2.5-3B-Instruct",
    load_in_4bit=True,
    load_in_8bit=False,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    temperature=0.0,
    top_p=0.9,
    top_k=40,
    max_new_tokens=2048,
    max_context_length=8192,
    device_map="auto",
    torch_dtype="auto",
    use_json_format=True,
    json_schema_validation=True,
    cache_dir=f"{CACHE_DIR}/qwen",
    enable_cache=True,
    max_retries=3,
    retry_backoff_factor=2.0,
)

print(f"Loading {RECOMMENDED_MODEL} with 4-bit quantization...")
print("This may take 1-3 minutes on first run (model download)...")

qwen_client = QwenClient(qwen_config)

# Test the model
print("\nTesting model with a simple prompt...")
test_response = qwen_client.generate_structured(
    task="fact_verification",
    prompt="Test: Return JSON with claim_id='test', claim_text='test', evidence_relationship='NEITHER', predicted_label='UNDETERMINED', confidence=0.0, evidence_ids=[], supporting_evidence=[], contradicting_evidence=[], reasoning='test', uncertainty_reason='test', model_identifier='test'",
)
print(f"Test response: success={test_response.success}, model={test_response.model}")
if test_response.error:
    print(f"Error: {test_response.error}")
print("\u2705 Qwen model loaded and tested successfully")

In [ ]:
# =============================================================================
# CELL 5: INITIALIZE PIPELINE & KNOWLEDGE BASE
# =============================================================================
# This initializes the complete TrumorGPT pipeline with GraphRAG

from trumourgpt.experiments.pipeline_qwen import TrumorGPTPipelineQwen, PipelineConfig
from trumourgpt.llm import QwenClientConfig

# Configure pipeline
pipeline_config = PipelineConfig(
    use_sentence_centrality=True,
    use_topic_textrank=True,
    use_graphrag=True,
    save_intermediate=False,
    output_dir=f"{RESULTS_DIR}/pipeline_qwen",
    qwen=QwenClientConfig(
        model_name=RECOMMENDED_MODEL,
        fallback_model="Qwen/Qwen2.5-3B-Instruct",
        load_in_4bit=True,
        temperature=0.0,
        top_p=0.9,
        max_new_tokens=2048,
        cache_dir=f"{CACHE_DIR}/qwen",
        enable_cache=True,
    ),
)

print("Initializing TrumorGPT pipeline with Qwen2.5...")
pipeline = TrumorGPTPipelineQwen(pipeline_config)
pipeline.initialize()

print("\u2705 Pipeline initialized successfully")
print(f"   KB subgraphs: {len(pipeline._components['retriever']._subgraphs)}")
print(f"   Total triples: {sum(len(t) for t in pipeline._components['retriever']._subgraph_triples.values())}")

In [ ]:
# =============================================================================
# CELL 6: TEST ON 3 DIAGNOSTIC CLAIMS
# =============================================================================
# These are the 3 mandatory diagnostic claims from the paper

DIAGNOSTIC_CLAIMS = [
    {
        "claim_id": "diagnostic_1",
        "text": "SARS-CoV-2 causes COVID-19.",
        "expected": "TRUE"
    },
    {
        "claim_id": "diagnostic_2",
        "text": "COVID-19 vaccines cause infertility.",
        "expected": "FALSE"
    },
    {
        "claim_id": "diagnostic_3",
        "text": "mRNA vaccines do not alter human DNA.",
        "expected": "TRUE"
    },
]

print("=" * 70)
print("TESTING ON 3 DIAGNOSTIC CLAIMS")
print("=" * 70)

results = []
for claim in DIAGNOSTIC_CLAIMS:
    print(f"\nProcessing: {claim['claim_id']}")
    print(f"Claim: {claim['text']}")
    print(f"Expected: {claim['expected']}")
    
    result = pipeline.process_claim(claim['text'], claim['claim_id'])
    
    pred = result.verification_result.predicted_label.value
    conf = result.verification_result.confidence
    rel = result.verification_result.evidence_relationship
    ev_count = len(result.verification_result.retrieved_evidence)
    time_sec = result.total_time
    
    status = "\u2705 PASS" if pred == claim['expected'] else "\u274c FAIL"
    
    print(f"Predicted: {pred} (confidence: {conf:.2f})")
    print(f"Evidence Relationship: {rel}")
    print(f"Evidence Count: {ev_count}")
    print(f"Time: {time_sec:.1f}s")
    print(f"Status: {status}")
    
    results.append({
        "claim_id": claim['claim_id'],
        "text": claim['text'],
        "expected": claim['expected'],
        "predicted": pred,
        "confidence": conf,
        "evidence_relationship": rel,
        "evidence_count": ev_count,
        "processing_time": time_sec,
        "correct": pred == claim['expected']
    })

# Summary
correct = sum(1 for r in results if r['correct'])
print(f"\n{'='*70}")
print(f"DIAGNOSTIC SUMMARY: {correct}/3 correct")
for r in results:
    status = "\u2705" if r['correct'] else "\u274c"
    print(f"  {status} {r['claim_id']}: {r['predicted']} (expected {r['expected']})")

In [ ]:
# =============================================================================
# CELL 7: EVALUATE ON 10 DEV CLAIMS
# =============================================================================
# Quick evaluation on 10 development claims to verify retrieval & verification

from trumourgpt.evaluation.dataset import PolitiFactDataset, DatasetConfig
from trumourgpt.evaluation.metrics import evaluate_binary, EvaluationMetrics
import random

# Load dev claims
settings = __import__('trumourgpt.config.settings', fromlist=['get_settings']).get_settings("config.yaml")
dataset = PolitiFactDataset(DatasetConfig(
    raw_path=settings.dataset.raw_path,
    processed_path=settings.dataset.processed_path,
    splits_path=settings.dataset.splits_path,
    target_size=settings.dataset.target_size,
    true_count=settings.dataset.true_count,
    false_count=settings.dataset.false_count,
    splits=settings.dataset.splits,
))
claims = dataset.load_processed()
dev_claims = dataset.get_split("dev")

# Sample 10 claims (deterministic)
random.seed(42)
sample_10 = random.sample(dev_claims, 10)

print("=" * 70)
print("EVALUATING ON 10 DEV CLAIMS")
print("=" * 70)

dev_results = []
for i, claim in enumerate(sample_10):
    print(f"\n[{i+1}/10] {claim.claim_id}: {claim.text[:80]}...")
    print(f"    True label: {claim.binary_label.value}")
    
    result = pipeline.process_claim(claim.text, str(claim.claim_id))
    
    pred = result.verification_result.predicted_label.value
    conf = result.verification_result.confidence
    rel = result.verification_result.evidence_relationship
    ev_count = len(result.verification_result.retrieved_evidence)
    time_sec = result.total_time
    
    correct = pred == claim.binary_label.value
    status = "\u2705" if correct else "\u274c"
    undet = " (UNDETERMINED)" if pred == "UNDETERMINED" else ""
    
    print(f"    Predicted: {pred}{undet} (conf: {conf:.2f}) {status}")
    print(f"    Evidence: {ev_count} graphs, Rel: {rel}, Time: {time_sec:.1f}s")
    
    dev_results.append({
        "claim_id": str(claim.claim_id),
        "text": claim.text,
        "true_label": claim.binary_label.value,
        "predicted": pred,
        "confidence": conf,
        "evidence_relationship": rel,
        "evidence_count": ev_count,
        "processing_time": time_sec,
        "correct": correct
    })

# Metrics
y_true = [r['true_label'] for r in dev_results]
y_pred = [r['predicted'] for r in dev_results]

# Only evaluate on determined predictions
determined = [(t, p) for t, p in zip(y_true, y_pred) if p != "UNDETERMINED"]
if determined:
    y_true_det, y_pred_det = zip(*determined)
    metrics = evaluate_binary(list(y_true_det), list(y_pred_det), "TRUE", "FALSE")
    print(f"\n{'='*70}")
    print(f"10-CLAIM DEV RESULTS:")
    print(f"  Total: 10")
    print(f"  Determined: {len(determined)} ({len(determined)/10*100:.0f}%)")
    print(f"  UNDETERMINED: {10 - len(determined)} ({(10-len(determined))/10*100:.0f}%)")
    print(f"  Accuracy (determined): {metrics.accuracy:.4f}")
    print(f"  Precision: {metrics.precision:.4f}")
    print(f"  Recall: {metrics.recall:.4f}")
    print(f"  F1: {metrics.f1:.4f}")
else:
    print(f"\n{'='*70}")
    print(f"10-CLAIM DEV RESULTS:")
    print(f"  All UNDETERMINED - check retrieval")

avg_time = sum(r['processing_time'] for r in dev_results) / 10
print(f"  Avg inference time: {avg_time:.1f}s")

# Save results
import json
with open(f"{RESULTS_DIR}/dev_10_results.json", "w") as f:
    json.dump(dev_results, f, indent=2)
print(f"\nResults saved to {RESULTS_DIR}/dev_10_results.json")

# =============================================================================
# DECISION POINT: Review 10-claim results before proceeding
# =============================================================================

**Check the following before proceeding to 100 claims:**

1. **Retrieval Coverage**: Are evidence graphs being retrieved? (evidence_count > 0)
2. **Verification Success**: Are claims getting TRUE/FALSE (not all UNDETERMINED)?
3. **Accuracy**: Is determined accuracy reasonable? (>50% on determined)
4. **Inference Time**: Is avg time acceptable? (< 60s per claim)
5. **Diagnostic Claims**: Did all 3 diagnostic claims pass?

**If satisfied, run Cell 8.** If not, debug retrieval/verification first.

**Do NOT use these results for model selection or prompt tuning.**

In [ ]:
# =============================================================================
# CELL 8: EVALUATE ON 100 DEV CLAIMS (with checkpointing)
# =============================================================================
# Full development set evaluation with crash-safe checkpoints

import json
import time
import sqlite3
from pathlib import Path
from datetime import datetime

# Checkpoint database
CHECKPOINT_DB = f"{CHECKPOINT_DIR}/dev_100_checkpoint.db"
RESULTS_FILE = f"{RESULTS_DIR}/dev_100_results.json"

def init_checkpoint_db():
    conn = sqlite3.connect(CHECKPOINT_DB)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS predictions (
            claim_id TEXT PRIMARY KEY,
            claim_text TEXT,
            true_label TEXT,
            predicted_label TEXT,
            confidence REAL,
            evidence_count INTEGER,
            evidence_relationship TEXT,
            reasoning TEXT,
            processing_time REAL,
            timestamp TEXT,
            model TEXT
        )
    """)
    conn.commit()
    return conn

def save_prediction(conn, record):
    conn.execute("""
        INSERT OR REPLACE INTO predictions VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        record['claim_id'], record['text'], record['true_label'],
        record['predicted'], record['confidence'], record['evidence_count'],
        record['evidence_relationship'], record['reasoning'],
        record['processing_time'], record['timestamp'], record['model']
    ))
    conn.commit()

def get_completed(conn):
    cursor = conn.execute("SELECT claim_id FROM predictions")
    return {row[0] for row in cursor.fetchall()}

# Initialize
conn = init_checkpoint_db()
completed = get_completed(conn)
 
# Get all dev claims
dev_claims = dataset.get_split("dev")
dev_claims_sorted = sorted(dev_claims, key=lambda c: c.claim_id)
 
print(f"Total dev claims: {len(dev_claims_sorted)}")
print(f"Already completed: {len(completed)}")
print(f"Remaining: {len(dev_claims_sorted) - len(completed)}")

# Process remaining
all_results = []
for claim in dev_claims_sorted:
    claim_id = str(claim.claim_id)
    
    if claim_id in completed:
        # Load from checkpoint
        cursor = conn.execute("SELECT * FROM predictions WHERE claim_id = ?", (claim_id,))
        row = cursor.fetchone()
        if row:
            cols = [d[0] for d in cursor.description]
            all_results.append(dict(zip(cols, row)))
        continue
    
    print(f"[{len(all_results)+1}/{len(dev_claims_sorted)}] {claim_id}: {claim.text[:60]}...")
    
    start = time.time()
    result = pipeline.process_claim(claim.text, claim_id)
    elapsed = time.time() - start
    
    pred = result.verification_result.predicted_label.value
    conf = result.verification_result.confidence
    rel = result.verification_result.evidence_relationship
    ev_count = len(result.verification_result.retrieved_evidence)
    reasoning = result.verification_result.reasoning
    
    record = {
        'claim_id': claim_id,
        'text': claim.text,
        'true_label': claim.binary_label.value,
        'predicted': pred,
        'confidence': conf,
        'evidence_count': ev_count,
        'evidence_relationship': rel,
        'reasoning': reasoning,
        'processing_time': elapsed,
        'timestamp': datetime.now().isoformat(),
        'model': RECOMMENDED_MODEL
    }
    
    save_prediction(conn, record)
    all_results.append(record)
    
    correct = pred == claim.binary_label.value
    status = "\u2705" if correct else ("\u2753" if pred == "UNDETERMINED" else "\u274c")
    print(f"    -> {pred} (conf: {conf:.2f}, ev: {ev_count}, time: {elapsed:.1f}s) {status}")
    
    # Periodic save to JSON
    if len(all_results) % 10 == 0:
        with open(RESULTS_FILE, 'w') as f:
            json.dump(all_results, f, indent=2)

# Final save
with open(RESULTS_FILE, 'w') as f:
    json.dump(all_results, f, indent=2)

conn.close()
print(f"\n\u2705 Completed {len(all_results)} dev claims")
print(f"Results saved to {RESULTS_FILE}")

In [ ]:
# =============================================================================
# CELL 9: ANALYZE 100 DEV CLAIM RESULTS
# =============================================================================

from trumourgpt.evaluation.metrics import evaluate_binary

# Load results
with open(f"{RESULTS_DIR}/dev_100_results.json", "r") as f:
    results = json.load(f)

y_true = [r['true_label'] for r in results]
y_pred = [r['predicted'] for r in results]

# Overall stats
total = len(results)
determined = [(t, p) for t, p in zip(y_true, y_pred) if p != "UNDETERMINED"]
undet_count = total - len(determined)

print("=" * 70)
print("100 DEV CLAIMS - FINAL ANALYSIS")
print("=" * 70)
print(f"Total claims: {total}")
print(f"Determined: {len(determined)} ({len(determined)/total*100:.1f}%)")
print(f"UNDETERMINED: {undet_count} ({undet_count/total*100:.1f}%)")

if determined:
    y_true_det, y_pred_det = zip(*determined)
    metrics = evaluate_binary(list(y_true_det), list(y_pred_det), "TRUE", "FALSE")
    print(f"\nDetermined Metrics:")
    print(f"  Accuracy: {metrics.accuracy:.4f}")
    print(f"  Precision: {metrics.precision:.4f}")
    print(f"  Recall: {metrics.recall:.4f}")
    print(f"  F1: {metrics.f1:.4f}")
    
    # Confusion matrix
    if metrics.confusion_matrix:
        cm = metrics.confusion_matrix
        print(f"\nConfusion Matrix (TRUE/FALSE):")
        print(f"              Pred TRUE  Pred FALSE")
        print(f"  True TRUE      {cm[0][0]:4d}       {cm[0][1]:4d}")
        print(f"  True FALSE     {cm[1][0]:4d}       {cm[1][1]:4d}")

# Overall accuracy (counting UNDETERMINED as wrong)
overall_correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
print(f"\nOverall Accuracy (UNDETERMINED=wrong): {overall_correct/total:.4f}")

# Per-label breakdown
for label in ["TRUE", "FALSE"]:
    label_true = [p for t, p in zip(y_true, y_pred) if t == label]
    if label_true:
        correct = sum(1 for p in label_true if p == label)
        undet = sum(1 for p in label_true if p == "UNDETERMINED")
        wrong = len(label_true) - correct - undet
        print(f"  {label}: {correct}/{len(label_true)} correct, {undet} undet, {wrong} wrong")

# Average inference time
avg_time = sum(r['processing_time'] for r in results) / total
print(f"\nAvg inference time: {avg_time:.1f}s")

# Evidence retrieval stats
ev_counts = [r['evidence_count'] for r in results]
print(f"\nEvidence retrieval:")
print(f"  Mean: {sum(ev_counts)/len(ev_counts):.1f}")
print(f"  Zero evidence: {sum(1 for c in ev_counts if c == 0)}")
print(f"  Max: {max(ev_counts)}")

# =============================================================================
# DECISION POINT: Review 100-claim results before test evaluation
# =============================================================================

**Check the following before proceeding to 400 held-out test claims:**

1. **Retrieval Coverage**: Is evidence being retrieved for most claims?
2. **Determination Rate**: Is UNDETERMINED rate acceptable? (< 50%)
3. **Determined Accuracy**: Is accuracy on determined claims reasonable?
4. **No Data Leakage**: Did you avoid looking at test labels?
5. **Diagnostic Claims**: Still passing?

**If satisfied, run Cell 10.** Otherwise, STOP and debug.

\u26a0\ufe0f **CRITICAL**: The 400 test claims are HELD-OUT. Do not use their results
for any model selection, prompt tuning, retrieval tuning, or debugging.
Run Cell 10 ONCE and report final metrics.

In [ ]:
# =============================================================================
# CELL 10: EVALUATE ON 400 HELD-OUT TEST CLAIMS
# =============================================================================
# FINAL EVALUATION - Run ONCE after dev approval

import json
import time
import sqlite3
from datetime import datetime

# Checkpoint database for test
TEST_CHECKPOINT_DB = f"{CHECKPOINT_DIR}/test_400_checkpoint.db"
TEST_RESULTS_FILE = f"{RESULTS_DIR}/test_400_results.json"

def init_test_db():
    conn = sqlite3.connect(TEST_CHECKPOINT_DB)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS predictions (
            claim_id TEXT PRIMARY KEY,
            claim_text TEXT,
            true_label TEXT,
            predicted_label TEXT,
            confidence REAL,
            evidence_count INTEGER,
            evidence_relationship TEXT,
            reasoning TEXT,
            processing_time REAL,
            timestamp TEXT,
            model TEXT
        )
    """)
    conn.commit()
    return conn

def save_test_prediction(conn, record):
    conn.execute("""
        INSERT OR REPLACE INTO predictions VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        record['claim_id'], record['text'], record['true_label'],
        record['predicted'], record['confidence'], record['evidence_count'],
        record['evidence_relationship'], record['reasoning'],
        record['processing_time'], record['timestamp'], record['model']
    ))
    conn.commit()

def get_test_completed(conn):
    cursor = conn.execute("SELECT claim_id FROM predictions")
    return {row[0] for row in cursor.fetchall()}

# Initialize
test_conn = init_test_db()
test_completed = get_test_completed(test_conn)
 
# Get test claims
test_claims = dataset.get_split("test")
test_claims_sorted = sorted(test_claims, key=lambda c: c.claim_id)
 
print(f"Total test claims: {len(test_claims_sorted)}")
print(f"Already completed: {len(test_completed)}")
print(f"Remaining: {len(test_claims_sorted) - len(test_completed)}")
print("\n\u26a0\ufe0f  THIS IS THE HELD-OUT TEST SET - RUN ONCE ONLY")
print("\u26a0\ufe0f  Do not use results for tuning or model selection")

test_results = []
for claim in test_claims_sorted:
    claim_id = str(claim.claim_id)
    
    if claim_id in test_completed:
        cursor = test_conn.execute("SELECT * FROM predictions WHERE claim_id = ?", (claim_id,))
        row = cursor.fetchone()
        if row:
            cols = [d[0] for d in cursor.description]
            test_results.append(dict(zip(cols, row)))
        continue
    
    print(f"[{len(test_results)+1}/{len(test_claims_sorted)}] {claim_id}: {claim.text[:60]}...")
    
    start = time.time()
    result = pipeline.process_claim(claim.text, claim_id)
    elapsed = time.time() - start
    
    pred = result.verification_result.predicted_label.value
    conf = result.verification_result.confidence
    rel = result.verification_result.evidence_relationship
    ev_count = len(result.verification_result.retrieved_evidence)
    reasoning = result.verification_result.reasoning
    
    record = {
        'claim_id': claim_id,
        'text': claim.text,
        'true_label': claim.binary_label.value,
        'predicted': pred,
        'confidence': conf,
        'evidence_count': ev_count,
        'evidence_relationship': rel,
        'reasoning': reasoning,
        'processing_time': elapsed,
        'timestamp': datetime.now().isoformat(),
        'model': RECOMMENDED_MODEL
    }
    
    save_test_prediction(test_conn, record)
    test_results.append(record)
    
    correct = pred == claim.binary_label.value
    status = "\u2705" if correct else ("\u2753" if pred == "UNDETERMINED" else "\u274c")
    print(f"    -> {pred} (conf: {conf:.2f}, ev: {ev_count}, time: {elapsed:.1f}s) {status}")
    
    # Periodic save
    if len(test_results) % 25 == 0:
        with open(TEST_RESULTS_FILE, 'w') as f:
            json.dump(test_results, f, indent=2)

# Final save
with open(TEST_RESULTS_FILE, 'w') as f:
    json.dump(test_results, f, indent=2)

test_conn.close()
print(f"\n\u2705 Completed {len(test_results)} test claims")
print(f"Results saved to {TEST_RESULTS_FILE}")

In [ ]:
# =============================================================================
# CELL 11: FINAL TEST SET ANALYSIS
# =============================================================================

from trumourgpt.evaluation.metrics import evaluate_binary

with open(f"{RESULTS_DIR}/test_400_results.json", "r") as f:
    test_results = json.load(f)

y_true = [r['true_label'] for r in test_results]
y_pred = [r['predicted'] for r in test_results]

total = len(test_results)
determined = [(t, p) for t, p in zip(y_true, y_pred) if p != "UNDETERMINED"]
undet_count = total - len(determined)

print("=" * 70)
print("FINAL TEST SET RESULTS (400 CLAIMS - HELD OUT)")
print("=" * 70)
print(f"Total claims: {total}")
print(f"Determined: {len(determined)} ({len(determined)/total*100:.1f}%)")
print(f"UNDETERMINED: {undet_count} ({undet_count/total*100:.1f}%)")

if determined:
    y_true_det, y_pred_det = zip(*determined)
    metrics = evaluate_binary(list(y_true_det), list(y_pred_det), "TRUE", "FALSE")
    print(f"\nDetermined Metrics:")
    print(f"  Accuracy: {metrics.accuracy:.4f}")
    print(f"  Precision: {metrics.precision:.4f}")
    print(f"  Recall: {metrics.recall:.4f}")
    print(f"  F1: {metrics.f1:.4f}")
    
    if metrics.confusion_matrix:
        cm = metrics.confusion_matrix
        print(f"\nConfusion Matrix:")
        print(f"              Pred TRUE  Pred FALSE")
        print(f"  True TRUE      {cm[0][0]:4d}       {cm[0][1]:4d}")
        print(f"  True FALSE     {cm[1][0]:4d}       {cm[1][1]:4d}")

# Overall accuracy (UNDETERMINED = wrong)
overall_correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
overall_acc = overall_correct / total
print(f"\nOVERALL BINARY ACCURACY (Paper target >=88%): {overall_acc:.4f} ({overall_acc*100:.2f}%)")
print(f"  Correct: {overall_correct}/{total}")

# Per-label breakdown
for label in ["TRUE", "FALSE"]:
    label_true = [p for t, p in zip(y_true, y_pred) if t == label]
    if label_true:
        correct = sum(1 for p in label_true if p == label)
        undet = sum(1 for p in label_true if p == "UNDETERMINED")
        wrong = len(label_true) - correct - undet
        print(f"  {label}: {correct}/{len(label_true)} correct, {undet} undet, {wrong} wrong")

# Average inference time
avg_time = sum(r['processing_time'] for r in test_results) / total
print(f"\nAvg inference time: {avg_time:.1f}s")
print(f"Total time: {sum(r['processing_time'] for r in test_results)/60:.1f} min")

# Evidence retrieval stats
ev_counts = [r['evidence_count'] for r in test_results]
print(f"\nEvidence retrieval:")
print(f"  Mean: {sum(ev_counts)/len(ev_counts):.1f}")
print(f"  Zero evidence: {sum(1 for c in ev_counts if c == 0)}")
print(f"  Max: {max(ev_counts)}")

# Save final summary
summary = {
    "model": RECOMMENDED_MODEL,
    "quantization": QUANTIZATION,
    "total_claims": total,
    "determined_claims": len(determined),
    "undetermined_claims": undet_count,
    "determination_rate": len(determined)/total,
    "overall_accuracy": overall_acc,
    "determined_accuracy": metrics.accuracy if determined else 0,
    "precision": metrics.precision if determined else 0,
    "recall": metrics.recall if determined else 0,
    "f1": metrics.f1 if determined else 0,
    "avg_inference_time_sec": avg_time,
    "timestamp": datetime.now().isoformat()
}

with open(f"{RESULTS_DIR}/test_400_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*70}")
print(f"SUMMARY SAVED TO {RESULTS_DIR}/test_400_summary.json")
print(f"=" * 70)

# =============================================================================
# SETUP INSTRUCTIONS
# =============================================================================

## How to Run This Notebook in Google Colab

### 1. Open in Colab
- Upload this `.ipynb` file to Google Colab (File -> Upload notebook)
- Or open directly from GitHub: `https://colab.research.google.com/github/YOUR_USERNAME/trumourgpt/blob/main/colab_trumourgpt_qwen.ipynb`

### 2. Enable GPU Runtime
- **Runtime** -> **Change runtime type** -> **Hardware accelerator: GPU**
- **Runtime** -> **Change runtime type** -> **GPU type: A100** (if available, else T4)
- Click **Save**

### 3. Run Cells Sequentially
- **Cell 1**: Setup & GPU Check - verifies VRAM, recommends model size
- **Cell 2**: Mount Google Drive - **click the link**, authorize, paste code
- **Cell 3**: Clone Repo & Install - **UPDATE `REPO_URL`** with your GitHub URL
- **Cell 4**: Load Qwen Model - downloads ~4-8GB model (first run takes 2-5 min)
- **Cell 5**: Initialize Pipeline - loads KB, builds GraphRAG index
- **Cell 6**: Test Diagnostic Claims - validates core functionality
- **Cell 7**: 10 Dev Claims - quick sanity check
- **STOP & REVIEW** - check metrics before proceeding
- **Cell 8**: 100 Dev Claims - with checkpointing (resumes if disconnected)
- **STOP & REVIEW** - check metrics before test evaluation
- **Cell 10**: 400 Test Claims - **HELD OUT**, run ONCE only
- **Cell 11**: Final Analysis - prints paper-target metrics

### 4. Resume After Disconnection
- Colab disconnects after ~90 min idle or ~12 hours max
- **Re-run Cells 1-5** to reload model & pipeline
- **Cell 8/10 will auto-resume** from SQLite checkpoint in Google Drive
- No work lost - predictions saved per-claim

### 5. Key Paths
- **Checkpoints**: `/content/drive/MyDrive/trumourgpt_checkpoints/`
- **Results**: `/content/drive/MyDrive/trumourgpt_results/`
- **Model Cache**: `/content/drive/MyDrive/trumourgpt_cache/`

### 6. Troubleshooting
- **OOM Error**: Cell 1 will auto-select 3B model if VRAM < 8GB
- **Slow Inference**: T4 GPU ~30-60s/claim, A100 ~5-15s/claim
- **Model Download Fails**: Check internet, try `huggingface-cli login`
- **Pipeline Errors**: Check logs, verify KB loaded in Cell 5

### 7. Expected Outputs
- `dev_10_results.json` - 10-claim quick evaluation
- `dev_100_results.json` - 100-claim dev evaluation
- `test_400_results.json` - 400-claim held-out test evaluation
- `test_400_summary.json` - Final metrics for paper comparison

### 8. Paper Target
- **Binary Accuracy >= 88%** on 600 PolitiFact health claims
- This notebook evaluates on 400 held-out test claims
- Dev claims (100) used only for retrieval/verification debugging

### 9. Important Notes
- **Never** use test labels for model selection or prompt tuning
- **Never** mix Gemini and Qwen results in same directory
- Qwen results saved to `trumourgpt_results/pipeline_qwen/`
- Gemini results remain in `experiments/results/batch_eval_*/`
- All randomness controlled by seed=42 for reproducibility